In [1]:
#%pip install -q optuna streamlit gradio scikit-learn xgboost torch plotly joblib pandas numpy ipython-autotime

In [2]:
%load_ext autotime

time: 110 µs (started: 2026-09-21 21:31:46 +05:30)


In [3]:
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.io as pio

# Optuna's plots are plotly figures. Pinning the renderer keeps them interactive in
# VS Code and Jupyter without embedding a 4.5 MB copy of plotly.js in this file.
pio.renderers.default = "plotly_mimetype+notebook_connected"

DATA = Path("data")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("future.no_silent_downcasting", True)
warnings.filterwarnings("ignore", category=FutureWarning)

import optuna as _optuna, sklearn as _sklearn, xgboost as _xgboost, torch as _torch

# Defaults, APIs and plot styles all move between releases. Pin what the lecture ran on.
print("versions        :", f"optuna {_optuna.__version__} · scikit-learn {_sklearn.__version__} "
      f"· xgboost {_xgboost.__version__} · torch {_torch.__version__}")
print("data files      :", sorted(p.name for p in DATA.glob("*.csv")))
print("artifacts so far:", sorted(p.name for p in ARTIFACTS.glob("*")) or "(empty — Part 5 fills this)")

versions        : optuna 4.9.0 · scikit-learn 1.7.2 · xgboost 3.2.0 · torch 2.10.0
data files      : ['cars24-car-price.csv', 'ticker_history.csv']
artifacts so far: ['baseline_model.joblib', 'model_card.json', 'price_model.joblib', 'study.db']
time: 4.05 s (started: 2026-09-21 21:31:53 +05:30)


In [4]:
cars = pd.read_csv(DATA / "cars24-car-price.csv")
print(cars.shape)
cars.head()

(19980, 11)


,full_name,selling_price,year,seller_type,km_driven,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Maruti Alto Std,1.20,2012.0,Individual,120000,Petrol,Manual,19.70,796.0,46.30,5.0
1,Hyundai Grand i10 Asta,5.50,2016.0,Individual,20000,Petrol,Manual,18.90,1197.0,82.00,5.0
2,Hyundai i20 Asta,2.15,2010.0,Individual,60000,Petrol,Manual,17.00,1197.0,80.00,5.0
3,Maruti Alto K10 2010-2014 VXI,2.26,2012.0,Individual,37000,Petrol,Manual,20.92,998.0,67.10,5.0
4,Ford Ecosport 2015-2021 1.5 TDCi Titanium BSIV,5.70,2015.0,Dealer,30000,Diesel,Manual,22.77,1498.0,98.59,5.0


time: 45.8 ms (started: 2026-09-21 21:32:35 +05:30)


In [5]:
print(f"skew        : {cars['selling_price'].skew():.2f}")
print(f"median      : {cars['selling_price'].median():.2f} lakhs")
print(f"99th pct    : {cars['selling_price'].quantile(0.99):.2f} lakhs")
print(f"max         : {cars['selling_price'].max():.2f} lakhs")
print(f"cars over 50: {(cars['selling_price'] > 50).sum()} of {len(cars)}")

skew        : 9.27
median      : 5.20 lakhs
99th pct    : 45.00 lakhs
max         : 395.00 lakhs
cars over 50: 150 of 19980
time: 2.6 ms (started: 2026-09-21 21:33:10 +05:30)


## Base line Model

In [6]:
encode_dict = {
    "fuel_type": {"Diesel": 1, "Petrol": 2, "CNG": 3, "LPG": 4, "Electric": 5},
    "transmission_type": {"Manual": 1, "Automatic": 2},
    "seller_type": {"Dealer": 1, "Individual": 2, "Trustmark Dealer": 3},
}

FEATURES = [
    "year",
    "seller_type",
    "km_driven",
    "fuel_type",
    "transmission_type",
    "mileage",
    "engine",
    "max_power",
    "seats",
]

df = cars.drop(columns=["full_name"]).replace(encode_dict).infer_objects(copy=False)
X, y = df[FEATURES], df["selling_price"]
X.head()

,year,seller_type,km_driven,fuel_type,transmission_type,mileage,engine,max_power,seats
0,2012.0,2,120000,2,1,19.70,796.0,46.30,5.0
1,2016.0,2,20000,2,1,18.90,1197.0,82.00,5.0
2,2010.0,2,60000,2,1,17.00,1197.0,80.00,5.0
3,2012.0,2,37000,2,1,20.92,998.0,67.10,5.0
4,2015.0,1,30000,1,1,22.77,1498.0,98.59,5.0


time: 19.5 ms (started: 2026-09-21 21:34:36 +05:30)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("train:", X_train.shape, " test:", X_test.shape)

train: (15984, 9)  test: (3996, 9)
time: 3.35 ms (started: 2026-09-21 21:34:59 +05:30)


In [8]:
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

baseline = XGBRegressor(random_state=42).fit(X_train, y_train)
pred = baseline.predict(X_test)

print(f"test R^2 : {r2_score(y_test, pred):.4f}")
print(f"test MAE : {mean_absolute_error(y_test, pred):.4f} lakhs")

test R^2 : 0.8749
test MAE : 1.0244 lakhs
time: 264 ms (started: 2026-09-21 21:36:03 +05:30)



```python
best = None
for n in [200, 400, 800]:
    for d in [4, 6, 8]:
        for lr in [0.03, 0.1, 0.3]:
            score = cv_mae(XGBRegressor(n_estimators=n, max_depth=d, learning_rate=lr))
            if best is None or score < best[0]:
                best = (score, n, d, lr)
```

```
All learning rates
        |
        +----------------------+
        |                      |
   mostly poor             promising
                         0.0001–0.01
                              |
                    +---------+---------+
                    |                   |
                 mediocre             good
                                  0.0004–0.004
                                         |
                                  +------+------+
                                  |             |
                               good         very good
                                         ~0.001

```

## Optuna Terminologies
1. Trial: A single execution of the objective function with a specific set of hyperparameters.
2. Study: A collection of trials that are executed to optimize the objective function.

In [10]:
# Defining one trial

from sklearn.model_selection import KFold, cross_val_score

CV = KFold(n_splits=3, shuffle=True, random_state=0)


def cv_mae(model):
    """Mean absolute error across 3 folds of the training set. Lower is better."""
    scores = cross_val_score(
        model, X_train, y_train, cv=CV, scoring="neg_mean_absolute_error"
    )
    return -scores.mean()


import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # otherwise every trial prints a line


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    return cv_mae(XGBRegressor(random_state=42, **params))

time: 691 µs (started: 2026-09-21 21:49:10 +05:30)


In [12]:
study = optuna.create_study(
    direction="minimize",                              # MAE: lower is better
    sampler=optuna.samplers.TPESampler(seed=7),        # seeded so this notebook reproduces
    study_name="milepost-price-demo",
)

study.optimize(objective, n_trials=12)

print(f"trials run : {len(study.trials)}")
print("best params:")
for k, v in study.best_params.items():
    print(f"  {k:18s} {v:.4f}" if isinstance(v, float) else f"  {k:18s} {v}")

trials run : 12
best params:
  n_estimators       212
  max_depth          7
  learning_rate      0.1317
  subsample          0.7831
  colsample_bytree   0.9909
  reg_lambda         0.0073
time: 25.5 s (started: 2026-09-21 21:50:01 +05:30)


In [13]:
N_TRIALS = 40
studies = {}

for name, sampler in [
    ("random", optuna.samplers.RandomSampler(seed=7)),
    ("TPE", optuna.samplers.TPESampler(seed=7)),
]:
    t0 = time.perf_counter()
    s = optuna.create_study(direction="minimize", sampler=sampler, study_name=f"price-{name}")
    s.optimize(objective, n_trials=N_TRIALS)
    studies[name] = {"study": s, "seconds": time.perf_counter() - t0}
    print(f"{name:7s} {N_TRIALS} trials  best cv MAE {s.best_value:.4f}  in {studies[name]['seconds']:.0f}s")   

random  40 trials  best cv MAE 1.0956  in 118s
TPE     40 trials  best cv MAE 1.0872  in 129s
time: 4min 7s (started: 2026-09-21 21:56:35 +05:30)


In [14]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
)

plot_optimization_history(studies["random"]["study"]).update_layout(
    title="RandomSampler — 40 trials", height=380
).show()
plot_optimization_history(studies["TPE"]["study"]).update_layout(
    title="TPESampler — 40 trials", height=380
).show()

time: 318 ms (started: 2026-09-21 22:00:59 +05:30)


### Optuna with Neural Networks

In [15]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# These batches are 256x9. Spread across every core, the thread synchronisation
# costs more than the arithmetic -- one thread is measurably faster here.
torch.set_num_threads(1)

from sklearn.preprocessing import StandardScaler

# A validation split carved out of the TRAINING data. The test set is still untouched.
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train.values, test_size=0.2, random_state=1)

# Trees split on thresholds, so feature scale is irrelevant to them -- nothing above this
# point was scaled. A network multiplies its inputs by weights, so scale matters a great
# deal here. Note where the scaler is fitted: on X_fit ONLY. Fitting it on all of X_train
# would let the validation rows influence the numbers the model is then judged against.
mlp_scaler = StandardScaler().fit(X_fit)

t_X_fit = torch.tensor(mlp_scaler.transform(X_fit), dtype=torch.float32)
t_y_fit = torch.tensor(y_fit, dtype=torch.float32).unsqueeze(1)
t_X_val = torch.tensor(mlp_scaler.transform(X_val), dtype=torch.float32)
t_y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

print("fit:", tuple(t_X_fit.shape), " val:", tuple(t_X_val.shape))

fit: (12787, 9)  val: (3197, 9)
time: 16.9 ms (started: 2026-09-21 22:11:16 +05:30)


In [16]:
def build_model(trial):
    layers = []
    n_in = t_X_fit.shape[1]

    n_layers = trial.suggest_int("n_layers", 1, 3)
    for i in range(n_layers):
        # These parameter names only exist on trials that got this far in the loop.
        n_out = trial.suggest_int(f"units_l{i}", 16, 128, log=True)
        dropout = trial.suggest_float(f"dropout_l{i}", 0.0, 0.4)
        layers += [nn.Linear(n_in, n_out), nn.ReLU(), nn.Dropout(dropout)]
        n_in = n_out

    layers.append(nn.Linear(n_in, 1))
    return nn.Sequential(*layers)

time: 595 µs (started: 2026-09-21 22:12:28 +05:30)


In [17]:
EPOCHS = 40
BATCH = 256


def mlp_objective(trial):
    model = build_model(trial)
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()
    n = len(t_X_fit)
    best_val = float("inf")

    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(n)
        for i in range(0, n, BATCH):
            idx = order[i : i + BATCH]
            optimizer.zero_grad()
            loss_fn(model(t_X_fit[idx]), t_y_fit[idx]).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_mae = loss_fn(model(t_X_val), t_y_val).item()
        best_val = min(best_val, val_mae)

        trial.report(val_mae, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val

time: 526 µs (started: 2026-09-21 22:14:14 +05:30)


In [18]:
from optuna.trial import TrialState

mlp = {}
for name, pruner in [
    ("no pruner", optuna.pruners.NopPruner()),
    ("MedianPruner", optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)),
]:
    t0 = time.perf_counter()
    s = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=7),
        pruner=pruner,
        study_name=f"mlp-{name}",
    )
    s.optimize(mlp_objective, n_trials=25)
    elapsed = time.perf_counter() - t0
    pruned = [t for t in s.trials if t.state == TrialState.PRUNED]
    mlp[name] = {
        "study": s,
        "seconds": elapsed,
        "pruned": len(pruned),
        "epochs trained": sum(len(t.intermediate_values) for t in s.trials),
        "best val MAE": s.best_value,
    }
    print(
        f"{name:13s} {elapsed:6.0f}s   pruned {len(pruned):2d}/25   "
        f"epochs {mlp[name]['epochs trained']:4d}/{25 * EPOCHS}   best {s.best_value:.4f}"
    )

no pruner         19s   pruned  0/25   epochs 1000/1000   best 1.2587
MedianPruner      15s   pruned  9/25   epochs  720/1000   best 1.2610
time: 34.3 s (started: 2026-09-21 22:21:00 +05:30)
